# Evaluation

mAP@k, failure analysis, efficiency benchmark, quantity verification sample.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd if (cwd / "dataset").exists() else cwd.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.evaluate import (
    run_map_evaluation,
    collect_failures,
    load_evaluation_set,
    load_food_products_cached,
    CandidateRetriever,
    run_efficiency_benchmark,
    run_quantity_verification_sample,
)

## 1. mAP@k (retrieval)

In [2]:
# Use 10 recipes for quick run; set max_recipes=50 for full evaluation
results = run_map_evaluation(k_values=[1, 5, 10], max_recipes=10)
for k in [1, 5, 10]:
    print(f"mAP@{k} = {results[f'mAP@{k}']:.4f}")
    print(f"  queries={results[f'stats@{k}']['n_queries']}, no_candidates={results[f'stats@{k}']['n_no_candidates']}")

mAP@1 = 0.9869
  queries=153, no_candidates=1
mAP@5 = 0.9912
  queries=153, no_candidates=1
mAP@10 = 0.9881
  queries=153, no_candidates=1


## 2. Failure analysis (unmatched ingredients)

In [3]:
eval_df = load_evaluation_set()
retriever = CandidateRetriever(load_food_products_cached(), use_food_filter=False)
failures = collect_failures(eval_df, retriever, max_recipes=20)
print(f"Unmatched ingredients: {len(failures)}")
pd.DataFrame(failures).head(15)

Unmatched ingredients: 1


,recipe_name,ingredient
0,Rava Rotti Recipe (Karnataka Style Semolina Fl...,gingeror


## 3. Efficiency

In [5]:
eff = run_efficiency_benchmark(n_recipes=3, use_reranker=True)
print("Latency (with re-ranker):", eff)

Latency (with re-ranker): {'n_recipes': 3, 'use_reranker': True, 'mean_latency_seconds': 72.92619226400001, 'total_seconds': 218.77857679200002, 'mean_ingredients_per_recipe': 14.333333333333334, 'mean_time_per_ingredient_seconds': 5.0878738788837214}


## 4. Quantity verification sample

In [6]:
qty = run_quantity_verification_sample(n_recipes=5, ingredients_to_check=["salt", "sugar"])
pd.DataFrame(qty)

,recipe_name,ingredient,quantity_needed,unit,cart_product,cart_quantity,cart_price
0,Fish Tandoori Recipe,salt,1.0,pinch,Himalayan - Pink Salt/Uppu,1,93.75
1,Karwar Style Khatkhate Recipe (Mixed Vegetable...,salt,1.0,to taste,Himalayan - Pink Salt/Uppu,1,93.75
2,Kerala Chicken Curry Recipe With Freshly Groun...,salt,1.0,to taste,Himalayan - Pink Salt/Uppu,1,93.75
